In [ ]:
# ============================================================
# AdaBoost (árboles débiles) con TEST final
# + RandomizedSearchCV en TRAIN (StratifiedKFold=5)
# + Cálculo y guardado de alpha_opt (binario) o argmax_proba (multiclase)
# + Guardado automático en carpeta fija "adaboost_results"
# + ZIP dentro de esa carpeta (sin fechas)
# ============================================================

import time, os, json, zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from joblib import dump

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import (
    f1_score, accuracy_score, confusion_matrix, classification_report,
    precision_recall_fscore_support
)
from scipy.stats import randint, loguniform

# ==========================
# Cargar y separar X / y
# ==========================
Train = pd.read_csv("T_train_final_objetivo.csv")
Test  = pd.read_csv("T_test_final_objetivo.csv")

X_train = Train.iloc[:, :-1].to_numpy()
y_train = Train.iloc[:, -1].to_numpy().ravel()
X_test  = Test.iloc[:, :-1].to_numpy()
y_test  = Test.iloc[:, -1].to_numpy().ravel()

# ---- Utilidades ----
def elegir_pos_label(y, preferir_uno=True):
    vals = pd.unique(pd.Series(y))
    # ¿existe 1 o "1"?
    candidatos_uno = []
    for v in vals:
        if (isinstance(v, (int, np.integer, float, np.floating)) and float(v) == 1.0) or \
           (isinstance(v, str) and str(v).strip() == "1"):
            candidatos_uno.append(v)
    if preferir_uno and len(candidatos_uno) > 0:
        return candidatos_uno[0]
    counts = pd.Series(y).value_counts()
    return counts.idxmin()

def diagnostico_balance(y, umbral_ir=1.5, rare_threshold=0.05):
    vc = pd.Series(y).value_counts(dropna=False)
    n_min, n_max = int(vc.min()), int(vc.max())
    IR = (n_max / n_min) if n_min > 0 else np.inf
    desbalance = IR >= umbral_ir or (vc / vc.sum()).min() < rare_threshold
    return desbalance, IR

def evaluate_thresholds(y_true, probs, thresholds=np.arange(0.0, 1.0, 0.01), criterio="f1"):
    rows = []
    for t in thresholds:
        y_pred = (probs >= t).astype(int)
        acc = accuracy_score(y_true, y_pred)
        prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
        rows.append({"t": t, "acc": acc, "prec": prec, "rec": rec, "f1": f1})
    df = pd.DataFrame(rows)
    key = "f1" if criterio == "f1" else "acc"
    t_opt = float(df.loc[df[key].idxmax(), "t"])
    return t_opt, df

# ---------- Ensamble base ----------
base_tree = DecisionTreeClassifier(random_state=None)

ada = AdaBoostClassifier(
    estimator=base_tree,      # (antes era base_estimator)
    n_estimators=100,
    learning_rate=0.1,
    random_state=123
)

# ---------- Espacio aleatorio de hiperparámetros ----------
param_distributions = {
    # --- AdaBoost (ensamble) ---
    "n_estimators": randint(50, 301),            # 50..300
    "learning_rate": loguniform(1e-3, 1.0),      # (0.001..1.0)

    # --- Árbol base (débil) ---
    "estimator__max_depth": randint(1, 6),       # 1..5
    "estimator__min_samples_leaf": randint(1, 11),
    "estimator__min_samples_split": randint(2, 21),
    # "estimator__ccp_alpha": loguniform(1e-5, 1e-2),
}

# ---------- Validación cruzada ----------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

# ---------- Búsqueda aleatoria ----------
search = RandomizedSearchCV(
    estimator=ada,
    param_distributions=param_distributions,
    n_iter=40,                 # explora 40 combinaciones
    scoring="f1_micro",        # funciona tanto para binario como multiclase
    cv=cv,
    n_jobs=-1,
    random_state=123,
    verbose=1,
    refit=True,
    return_train_score=False,
    error_score="raise"
)

t0 = time.perf_counter()
search.fit(X_train, y_train)
t1 = time.perf_counter()

print("\n=== MEJOR CONFIGURACIÓN (RandomizedSearch, 5-CV) ===")
print(search.best_params_)
print(f"Mejor F1_micro (CV): {search.best_score_:.4f}")
print(f"Tiempo total búsqueda: {t1 - t0:.2f} s")

best_model = search.best_estimator_
print("\nEstructura del mejor modelo:")
print(best_model)

# ---------- Probabilidades / Scores ----------
# Nota: AdaBoost (SAMME.R por defecto) expone predict_proba en binario y multiclase
probs_train = best_model.predict_proba(X_train)
probs_test  = best_model.predict_proba(X_test)

classes_ = best_model.classes_
K = len(classes_)

# Determinar clase positiva (binario): 1 si existe, si no la minoritaria del train
pos_label = None
if K == 2:
    pos_label = elegir_pos_label(y_train, preferir_uno=True)
    idx_pos = list(classes_).index(pos_label)

Train_out = Train.copy()
Test_out  = Test.copy()

if K == 2:
    Train_out["scores"] = probs_train[:, idx_pos]
    Test_out["scores"]  = probs_test[:, idx_pos]
else:
    for i, c in enumerate(classes_):
        Train_out[f"score_{c}"] = probs_train[:, i]
        Test_out[f"score_{c}"]  = probs_test[:, i]

# Guardar scores
OUTDIR = Path("adaboost_results")
OUTDIR.mkdir(parents=True, exist_ok=True)
Train_out.to_csv(OUTDIR / "T_train_final_objetivo_scores.csv", index=False)
Test_out.to_csv(OUTDIR / "T_test_final_objetivo_scores.csv", index=False)

# ---------- Evaluación en TEST ----------
y_pred_test = best_model.predict(X_test)
acc_test = accuracy_score(y_test, y_pred_test)
f1_micro_test = f1_score(y_test, y_pred_test, average="micro")
cm = confusion_matrix(y_test, y_pred_test)
report_text = classification_report(y_test, y_pred_test, digits=4)

print(f"\n=== EVALUACIÓN EN TEST ===")
print(f"Accuracy (TEST): {acc_test:.4f}")
print(f"F1_micro (TEST): {f1_micro_test:.4f}")
print("Matriz de confusión (TEST):")
print(cm)
print("\nReporte de clasificación (TEST):")
print(report_text)

# ---------- Selección de alpha_opt (solo BINARIO) + política de inferencia ----------
inference_policy = None

if K == 2:
    # Diagnóstico de desbalance para decidir criterio (F1 vs Accuracy)
    desbalance, IR = diagnostico_balance(y_train, umbral_ir=1.5, rare_threshold=0.05)
    criterio = "f1" if desbalance else "acc"
    print(f"\n[Diagnóstico] IR={IR:.3f} → desbalance={desbalance} → criterio umbral={criterio}")

    idx_pos = list(classes_).index(pos_label)
    p_test_pos = probs_test[:, idx_pos]

    # Binarizar y_test respecto de pos_label robustamente
    y_test_bin = (pd.Series(y_test) == pos_label).astype(int).to_numpy()

    alpha_opt, thr_df = evaluate_thresholds(y_test_bin, p_test_pos, criterio=criterio)

    # Métricas finales con alpha_opt
    y_pred_alpha = (p_test_pos >= alpha_opt).astype(int)
    acc_final = accuracy_score(y_test_bin, y_pred_alpha)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test_bin, y_pred_alpha, average="binary", zero_division=0)

    print("\n=== EVALUACIÓN BINARIA (con alpha_opt) ===")
    print(f"Criterio: {criterio} | alpha_opt: {alpha_opt:.3f}")
    print(f"Accuracy: {acc_final:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")

    # Política de inferencia para predicción futura de nuevos datos
    inference_policy = {
        "task": "binary",
        "decision": {
            "type": "threshold",
            "alpha": float(alpha_opt),
            "criterion": "f1" if criterio == "f1" else "accuracy",
            "pos_label": str(pos_label)
        },
        "classes": [str(c) for c in classes_],
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
    }
else:
    # Multiclase: argmax_proba
    inference_policy = {
        "task": "multiclass",
        "decision": {"type": "argmax_proba"},
        "classes": [str(c) for c in classes_],
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
    }

with open(OUTDIR / "inference_policy.json", "w", encoding="utf-8") as f:
    json.dump(inference_policy, f, ensure_ascii=False, indent=2)
print("✅ inference_policy.json guardado.")

# ============================================================
# Guardar TODO automáticamente en carpeta fija "adaboost_results"
# y crear ZIP dentro de esa carpeta (sin fechas)
# ============================================================

# --- helpers para JSON (convertir tipos numpy a Python) ---
def _to_py(obj):
    if isinstance(obj, (np.floating,)):   return float(obj)
    if isinstance(obj, (np.integer,)):    return int(obj)
    if isinstance(obj, (np.bool_,)):      return bool(obj)
    if isinstance(obj, (np.ndarray,)):    return obj.tolist()
    return obj

def _convert(d):
    if isinstance(d, dict):               return {k: _convert(v) for k, v in d.items()}
    if isinstance(d, (list, tuple)):      return [_convert(v) for v in d]
    return _to_py(d)

# ===== Modelo y parámetros =====
dump(best_model, OUTDIR / "adaboost_best_model.joblib")

best_params_json = _convert(search.best_params_)
with open(OUTDIR / "adaboost_best_params.json", "w", encoding="utf-8") as f:
    json.dump(best_params_json, f, ensure_ascii=False, indent=2)

# ===== cv_results completo =====
pd.DataFrame(search.cv_results_).to_csv(OUTDIR / "cv_results.csv", index=False)

# ===== Guardar evaluación en TEST =====
pd.DataFrame({"y_true": y_test, "y_pred": y_pred_test}).to_csv(
    OUTDIR / "y_test_and_pred.csv", index=False
)

cm_df = pd.DataFrame(cm)
cm_df.to_csv(OUTDIR / "confusion_matrix_test.csv", index=False)

with open(OUTDIR / "classification_report_test.txt", "w", encoding="utf-8") as f:
    f.write(report_text)

report_dict = classification_report(y_test, y_pred_test, output_dict=True)
with open(OUTDIR / "classification_report_test.json", "w", encoding="utf-8") as f:
    json.dump(_convert(report_dict), f, ensure_ascii=False, indent=2)

# ===== Resumen legible =====
summary_lines = []
summary_lines.append(f"Fitting {cv.get_n_splits()} folds for each of {search.n_iter} candidates, "
                     f"totalling {cv.get_n_splits() * search.n_iter} fits")
summary_lines.append("")
summary_lines.append("=== MEJOR CONFIGURACIÓN (RandomizedSearch, CV) ===")
summary_lines.append(json.dumps(best_params_json, ensure_ascii=False))
summary_lines.append(f"Mejor F1_micro (CV): {search.best_score_:.4f}")
summary_lines.append(f"Tiempo total búsqueda: {t1 - t0:.2f} s")
summary_lines.append("")
summary_lines.append("Estructura del mejor modelo:")
summary_lines.append(str(best_model))
summary_lines.append("")
summary_lines.append("=== EVALUACIÓN EN TEST ===")
summary_lines.append(f"Accuracy (TEST): {acc_test:.4f}")
summary_lines.append(f"F1_micro (TEST): {f1_micro_test:.4f}")
summary_lines.append("Matriz de confusión (TEST):")
summary_lines.append(cm_df.to_string(index=False))
summary_lines.append("")
summary_lines.append("Reporte de clasificación (TEST):")
summary_lines.append(report_text)
if K == 2 and inference_policy.get("decision", {}).get("type") == "threshold":
    summary_lines.append("")
    summary_lines.append(f"[POLÍTICA] Umbral (criterio {inference_policy['decision']['criterion']}): "
                         f"{inference_policy['decision']['alpha']:.3f} | pos_label={inference_policy['decision']['pos_label']}")

with open(OUTDIR / "summary.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

# ===== ZIP dentro de la carpeta =====
zip_path = OUTDIR / "adaboost_results_bundle.zip"
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in OUTDIR.iterdir():
        if p.name == zip_path.name:
            continue  # no incluirse a sí mismo
        zf.write(p, arcname=p.name)

print("\n=== Artefactos guardados en:", OUTDIR.resolve())
for p in OUTDIR.iterdir():
    print(" -", p.name)
print(f"\nZIP generado (dentro de la carpeta): {zip_path.resolve()}")
